# Reproducing Komori et al. (2026) from DANDI:001425: decoding of spontaneous pupil dilation

**Paper.** Komori Y, Mizusaki K, Tomita K, Yoneda T, Matoba K, Morita T (2026). Mesoscopic cortical
activities associated with pupil-linked perceptions inferred via explainable machine learning.
bioRxiv, [doi:10.64898/2026.05.27.726399](https://doi.org/10.64898/2026.05.27.726399). The claims
file (`claims/komori-2026.yaml`) was written from the preprint posted June 1, 2026, with its
Methods, legends and Supplementary Figures 3 to 6.

**Dandiset.** [DANDI:001425](https://dandiarchive.org/dandiset/001425/0.250705.0947), version
0.250705.0947 ("BraiDyn-BC: mouse cued-lever-pull task dataset", Kondo et al. 2025). The paper
cites the dandiset without a version. Two versions are published, 0.250514.0602 (1814 assets) and
0.250705.0947 (1838 assets); the most recent is pinned, as the protocol asks when no version is
cited. The paper was posted in 2026, after both, so either could be the one it read. The data check
shows that 6 of the 9 files read here are the same assets in both versions; the other 3 were
replaced in the later version, and the earlier ones were not compared.
The paper uses only the resting-state recordings of this dandiset; its own recordings
(somatosensory and visual stimulation) are not deposited.

**What is tested.** The six resting-state claims. The paper's pipeline is rebuilt from its
Methods: hemodynamic correction, SVD and JADE ICA of the cortical images, pupil labels around the
dilation peak of each 40 s epoch, a GRU decoder with five-fold cross-validation and null models
trained on labels shuffled within epochs. The open details R1 to R10 of the claims file's opening
comment are followed as it names them; where several readings are named, the notebook runs the
main one and as many others as time allows, and says which were not run.

This notebook tests the three decoding claims; the companion notebook
`reproduce_komori_2026_features.ipynb` tests the IC count, the window, the VIF and the SHAP claims.

| Claim | Notebook | Status |
|---|---|---|
| rest-decoding-above-null | decoding | tested (one variant, R4 b, in the features notebook) |
| rest-loss-accuracy-vs-null | decoding | tested |
| rest-needs-0-1-hz | decoding | tested |
| rest-ic-count-and-window | features | tested |
| rest-vif-below-10 | features | tested |
| rest-shap-medial-anterior-first | features | tested |
| fig1-*, fig2-*, fig3-*, fig4-shap-*, fig4-mapped-shap-regions, supp2-*, supp4-*, supp8-*, supp10-* (18 claims) | none | blocked: the stimulation recordings are not deposited |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

import pupilwf as pw

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
assets = pw.list_assets()
rows = pw.resting_table(assets)
chosen = pw.choose_sessions(rows)          # R1 (a): three mice, first three resting sessions each
chosen_rows = [r for mouse in chosen for r in mouse]
sessions = pw.load_sessions(chosen_rows)
print(f"streamed and reduced {len(sessions)} sessions from DANDI:{pw.DANDISET} version {pw.VERSION} "
      f"in {time.time() - started:.0f} s")

In [ ]:
t0 = time.time()
D = pw.decompose(sessions, hemo="fit_hp_subtract_raw")       # R2 (a), main reading
comp = pw.compactness(D["maps"], D["mask"])
# features for VIF-based selection: every frame of the 13 epochs, 0.1 Hz low-pass (R6)
def vif_frames(ics, cutoff=0.1):
    out = []
    for s, ic in zip(sessions, ics):
        a, b = pw.epochs_of(s["t"])[0][0], pw.epochs_of(s["t"])[-1][1]
        out.append(pw.normalise_features(ic, cutoff)[a:b])
    return np.vstack(out)
F01 = vif_frames(D["ics"])
keep10, removed10 = pw.select_ics(F01, comp, 10, "kept")
print(f"decomposition in {time.time() - t0:.0f} s; kept ICs {keep10}")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
tab = pd.DataFrame(rows)
print("Resting-state NWB files in the dandiset:", len(tab), "from", tab.subject.nunique(), "mice")
print("Files per mouse:", tab.groupby("subject").size().value_counts().sort_index().to_dict(),
      "(number of files: number of mice)")
print("Files without behaviour (widefield only, no pupil):", int((~tab.behavior).sum()))
print(tab[~tab.behavior][["subject", "day"]].to_string(index=False))
print("Paper, Methods, Resting state: 25 mice, '2-3 experiments per animal'.")
print("\nChosen (R1 a): ", [(r["subject"], r["day"], r["date"]) for r in chosen_rows])
old = {a["path"]: a["asset_id"] for a in pw.list_assets("0.250514.0602")}
print("Chosen files with the same asset (same content) in version 0.250514.0602:",
      sum(old.get(r["path"]) == r["asset_id"] for r in chosen_rows), "of", len(chosen_rows))

In [ ]:
info = pd.DataFrame([{
    "subject": s["subject"], "day": s["day"], "frames": s["n_frames"],
    "rate_hz": round(1 / np.median(np.diff(s["t"])), 3), "duration_s": round(s["t"][-1], 1),
    "uv_after_blue_ms": round(1000 * s["t_uv_offset"], 1), "pupil_on_img_clock": s["pupil_t_equal"],
    "dff_on_img_clock": s["dff_t_equal"], "pupil_nan": s["pupil_nan"],
    "pupil_px_max_deposited": float(f"{np.nanmax(s['pupil_filled']):.3g}"), "pupil_replaced": s["pupil_replaced"],
    "pupil_px_min": round(float(s["pupil"].min()), 1), "pupil_px_median": round(float(np.median(s["pupil"])), 1),
    "pupil_px_max": round(float(s["pupil"].max()), 1),
    "kp_width_bad": s["pupil_kp_bad"], "r_diam_kp": round(float(np.corrcoef(s["pupil"], s["pupil_kp"])[0, 1]), 3),
    "cortex_px": s["n_pixels"], "svd34_var": round(s["svd_var_explained"], 4),
    "raw_blue": round(s["raw_mean"][0]), "raw_uv": round(s["raw_mean"][1]),
    "read_s": round(s["read_s"])} for s in sessions])
print(info.to_string(index=False))
print("\nPaper (Methods, Resting state): 60 Hz acquisition alternating 405/470 nm, 36,000 frames per 600 s "
      "session, i.e. 18,000 frames and 30 Hz per channel; frames 288 x 288 after down-sampling.")
print("Frames per channel:", sorted(set(info.frames)), "; frame shape in the files: 288 x 288 (binned 2x2 here)")
print("Hemodynamic regression slope of F470 on F405 (34 components per session): median",
      round(float(np.median(np.concatenate([s["coef"][:, 0] for s in sessions]))), 3),
      "range", np.round(np.percentile(np.concatenate([s["coef"][:, 0] for s in sessions]), [0, 100]), 2))
print("Pupil columns above: the deposited maximum, the frames replaced as failed detections (helper, clean_pupil), and "
      "the range after replacement. r_diam_kp: correlation of the cleaned diameter with the keypoint width (R3 b).")
for r in chosen_rows[:2]:
    print("Channel check,", r["subject"], r["day"], pw.channel_check(r))
f = pw.open_nwb(chosen_rows[0]["asset_id"], block_mb=2)
for plane in ("ImagingPlane_blue", "ImagingPlane_UV"):
    g = f["general/optophysiology/" + plane]
    print(plane, "excitation_lambda", g["excitation_lambda"][()], "imaging_rate", g["imaging_rate"][()],
          "indicator", g["indicator"][()].decode(), "grid_spacing", g["grid_spacing"][:])
print("acquisition/widefield_blue description:", f["acquisition/widefield_blue"].attrs["description"])
print("dFF_B description:", f["processing/ophys/DfOverF/dFF_B"].attrs["description"])
print("pupil diameter unit:", f["processing/behavior/pupil_tracking/diameter/data"].attrs["unit"])
print("genotype:", f["general/subject/genotype"][()].decode())
f.close()

Every session holds 18,000 frames per channel at 29.95 Hz over 600 s, as the paper describes
(36,000 frames at 60 Hz, alternating). The pupil diameter and the deposited dF/F are on the
imaging timestamps exactly. The deposited pupil diameter has failed detections that the files do
not flag: in VG1-GC#51 day0 and VG1-GC#53 day1 its maximum is 2.88e21 and 1.05e11 pixels, and
148 and 57 frames there are above three times the median (besides 768 and 269 missing values);
these are replaced before any label is made. The channel check shows the calcium signal in
`widefield_blue` (dF/F SD 0.050 and 0.028 against 0.012 and 0.014 in `widefield_UV`), as its
description says, but the imaging-plane metadata give 405 nm for the blue plane and 470 nm for the
UV plane: the excitation wavelengths are swapped in the metadata.

In [ ]:
# Recordings deposited twice: identical pupil traces or identical mean fluorescence would show it.
P = np.array([s["pupil"][:18000] for s in sessions])
C = np.corrcoef(P)
np.fill_diagonal(C, np.nan)
print("Largest pupil-trace correlation between two different sessions:", round(float(np.nanmax(C)), 3))
means = [s["raw_mean"] for s in sessions]
print("Distinct (blue, UV) mean raw intensities:", len(set(means)), "of", len(means))

In [ ]:
# Clock check. The pupil and the imaging share the timestamps (above). Whether the pupil trace is
# aligned with the imaging in fact is checked with the cross-correlation between the pupil and the
# mean cortical activity (deposited hemodynamics-corrected dF/F, averaged over the 44 ROIs): arousal
# links the two, and a misaligned trace would move the peak. Control: the same with the pupil
# circularly shifted by 100 to 500 s.
lags = np.arange(-300, 301, 3)
rows_clk = []
for s in sessions:
    p = pw.butter(s["pupil"], 0.2)
    p = (p - p.mean()) / p.std()
    g = pw.butter(np.nanmean(s["dff_deposited"], axis=1), 0.2)
    g = (g - g.mean()) / g.std()
    cc = np.array([np.mean(p[max(0, -l):len(p) - max(0, l)] * g[max(0, l):len(g) - max(0, -l)]) for l in lags])
    sh = []
    for k in range(5):
        q = np.roll(p, 3000 + 3000 * k)
        sh.append(np.max(np.abs([np.mean(q[max(0, -l):len(q) - max(0, l)] * g[max(0, l):len(g) - max(0, -l)]) for l in lags[::10]])))
    i = int(np.argmax(np.abs(cc)))
    rows_clk.append(dict(subject=s["subject"], day=s["day"], peak_r=round(float(cc[i]), 3),
                         lag_s=round(float(lags[i] / pw.FS), 2), r_at_0=round(float(cc[lags == 0][0]), 3),
                         shifted_max_abs_r=round(float(np.max(sh)), 3)))
print("Positive lag: cortical activity follows the pupil.")
print(pd.DataFrame(rows_clk).to_string(index=False))

The pupil and the mean cortical activity correlate in every session, with the pupil lagging
cortical activity by 1.1 to 2.3 s in seven of nine sessions (5.4 and 7.6 s in VG1-GC#52 day7 and
day1). The peak correlation exceeds the largest one with the pupil circularly shifted in all nine
sessions, narrowly in VG1-GC#52 day1 (0.210 against 0.206) and VG1-GC#53 day1 (0.187 against
0.169). This is consistent with the pupil and the imaging sharing a clock; it does not prove it,
and the clock of those two sessions is the least supported.

In [ ]:
print("Common cortical grid points (atlas grid, step", pw.GRID_STEP, "atlas pixels):", int(D["mask"].sum()))
print("SVD components available:", D["n_svd"], "(paper: 400; the corrected data of each recording have rank 34)")
print("Cumulative variance at 10, 30, 200 components:", np.round(D["cum_var"][[9, 29, min(199, D['n_svd'] - 1)]], 4),
      "(paper: > 99.5% at 200)")
print("Frames used to estimate the decomposition (every fifth 40 s epoch):", D["n_sel_frames"])
print("IC compactness (share of squared weight in the largest supra-half-maximum region):")
print(pd.Series(comp).round(3).to_string())
v30 = pw.vif(F01)
print("VIF of all 30 ICs (0.1 Hz features): max", round(float(v30.max()), 2))
print("Kept for 10 ICs (reading a):", keep10, "removed for VIF:", removed10,
      "max VIF of the kept:", round(float(pw.vif(F01[:, keep10]).max()), 2))
roi = sessions[0]["roi_atlas"]
names = sessions[0]["roi_names"]
fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for ax, j in zip(axes.flat, keep10):
    m = D["maps"][j]
    lim = np.nanmax(np.abs(m))
    ax.imshow(m, cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.contour(roi.any(axis=0), [0.5], colors="k", linewidths=0.5)
    ax.set_title(f"IC {j}  comp {comp[j]:.2f}", fontsize=9)
    ax.axis("off")
plt.suptitle("Spatial maps of the 10 kept ICs on the common atlas grid (colour limits: +/- max |weight|)")
plt.tight_layout()
plt.show()

In [ ]:
ds_main = pw.make_dataset(sessions, D["ics"], keep10, cutoff=0.1)
lab = pd.DataFrame(ds_main["info"])
print("Epochs per recording:", lab.groupby("rec").size().unique(), "; samples per epoch window:", sorted(lab.n.unique()))
print("Positive frames per epoch window (of", lab.n.max(), "), by recording:")
print(lab.groupby(["subject", "day"]).n_pos.describe()[["mean", "min", "50%", "max"]].round(1).to_string())
print("Epochs whose window has no positive frame:", int((lab.n_pos == 0).sum()), "of", len(lab),
      "; all frames positive:", int((lab.n_pos == lab.n).sum()))
print("Samples:", ds_main["X"].shape, "positive share", round(float(ds_main["y"].mean()), 3))
print("Feature range:", float(ds_main["X"].min()), float(ds_main["X"].max()))
lab5 = pd.DataFrame(pw.make_dataset(sessions, D["ics"], keep10, 0.1, selection="rule")["info"])
print("Epochs kept by the selection rule R5 (b):", int(lab5.keep.sum()), "of", len(lab5))

With the whole recording's minimum and maximum, the labels are mostly constant within an epoch:
42 of 117 epoch windows hold no positive frame and 37 are positive throughout, and VG1-GC#53 day1
has no positive frame at all. The selection rule R5 (b) keeps 17 of 117 epochs.

## rest-decoding-above-null

> A GRU decoder of independent components of resting cortical activity, low-pass filtered at 0.1 Hz,
> predicts spontaneous pupil dilation with a validation ROC-AUC significantly higher than that of
> null models trained on within-session shuffled labels. (Expected: model 0.82 +/- 0.082, null
> 0.61 +/- 0.061; paired t-test p < 0.05.)

Main reading combination: R1 a (3 mice x 3 sessions), R2 a (paper's correction on the frames,
coefficients applied to the measured signals), R3 a (deposited diameter), R4 a (samples in the
window), R5 a (no epoch selection), R6 (10 ICs, reading a), R7 a (folds of epochs), R8 a (training
labels shuffled), R9, at the claim's seed. Then the seeds plus 1000 to 4000, then the other readings
one at a time.

In [ ]:
SEED1 = pw.seed_of("rest-decoding-above-null")
t0 = time.time()
cv_main, models_main = pw.run_cv(ds_main, SEED1, keep_models=True)
cols = ["fold", "n_train", "n_val", "pos_val", "model_epoch", "model_train_auc", "model_val_auc", "null_val_auc",
        "model_val_loss", "null_val_loss", "model_val_acc", "null_val_acc"]
print(cv_main[cols].round(3).to_string(index=False))
r = pw.paired(cv_main, "val_auc")
print(f"\nvalidation AUC: model {r['model']:.3f}, null {r['null']:.3f}, paired t = {r['t']:.2f}, p = {r['p']:.3g}  "
      f"(expected 0.82 +/- 0.082 and 0.61 +/- 0.061)")
print(f"model AUC - null AUC per fold: {np.round(cv_main.model_val_auc - cv_main.null_val_auc, 3).tolist()}")
print(f"training AUC of the models: mean {cv_main.model_train_auc.mean():.3f}")
print(f"({time.time() - t0:.0f} s)")

In [ ]:
bym = pw.auc_by_mouse(ds_main, models_main)
print("Validation AUC by mouse (mean over folds):")
print(bym.groupby("mouse")[["model_auc", "null_auc"]].mean().round(3).to_string())

In [ ]:
seed_rows = []
cv_by_seed = {0: cv_main}
for off in (1000, 2000, 3000, 4000):
    cv_by_seed[off] = pw.run_cv(ds_main, SEED1 + off)
for off, cv in cv_by_seed.items():
    r = pw.paired(cv, "val_auc")
    seed_rows.append(dict(seed=f"claim+{off}", model_auc=r["model"], null_auc=r["null"], t=r["t"], p=r["p"],
                          above=bool(r["p"] < 0.05 and r["model"] > r["null"]),
                          within_tol=bool(abs(r["model"] - 0.82) <= 0.082)))
print(pd.DataFrame(seed_rows).round(4).to_string(index=False))

In [ ]:
# Other readings, one at a time, at the claim's seed.
def variant(name, sess=sessions, ics=None, keep=None, ds_kw=None, cv_kw=None):
    t0 = time.time()
    ds = pw.make_dataset(sess, ics if ics is not None else D["ics"], keep if keep is not None else keep10,
                         **({"cutoff": 0.1} | (ds_kw or {})))
    cv = pw.run_cv(ds, SEED1, **(cv_kw or {}))
    out = {"reading": name, "samples": len(ds["y"]), "pos": round(float(ds["y"].mean()), 3)}
    for m in ("val_auc", "val_loss", "val_acc"):
        r = pw.paired(cv, m)
        out[m + "_model"], out[m + "_null"], out[m + "_p"] = r["model"], r["null"], r["p"]
    out["s"] = round(time.time() - t0)
    return out, cv

D_hp = pw.decompose(sessions, hemo="hp_corrected")
comp_hp = pw.compactness(D_hp["maps"], D_hp["mask"])
keep_hp, _ = pw.select_ics(vif_frames(D_hp["ics"]), comp_hp, 10, "kept")
var_rows, var_cv = [], {}
for name, kw in [
        ("R2: corrected signal kept high-passed", dict(ics=D_hp["ics"], keep=keep_hp)),
        ("R3 b: keypoint width as pupil", dict(ds_kw={"pupil_key": "pupil_kp"})),
        ("R5 b: epoch selection rule", dict(ds_kw={"selection": "rule"})),
        ("R7 b: folds of recordings (pooled over mice)", dict(cv_kw={"fold_unit": "recording"})),
        ("R8 b: null shuffles validation labels too", dict(cv_kw={"null_scope": "both"}))]:
    o, cv = variant(name, **kw)
    var_rows.append(o)
    var_cv[name] = cv
vt = pd.DataFrame(var_rows)
print(vt[["reading", "samples", "pos", "val_auc_model", "val_auc_null", "val_auc_p", "s"]].round(4).to_string(index=False))
print("Folds with an AUC (R7 b; a fold whose validation recordings hold no positive label has none):",
      int(np.isfinite(var_cv["R7 b: folds of recordings (pooled over mice)"].model_val_auc).sum()), "of 5")

**Result: not reproduced under the main reading (see the R4 b variant in the features notebook,
which decides the outcome).** Under the main reading combination the models reach a mean validation
AUC of 0.686 against 0.708 for the null models (paired t = -1.02, p = 0.364), below the expected
0.82 +/- 0.082. No other seed gives a significant difference (p = 0.10 to 0.90, model means 0.678
to 0.705), and none of the other readings does: corrected signal kept high-passed 0.657 against
0.651 (p = 0.90), keypoint pupil 0.641 against 0.641 (p = 1.00), epoch selection 0.639 against
0.644 (p = 0.95), folds of recordings 0.581 against 0.611 over the 4 folds with an AUC (p = 0.55;
the fifth holds only VG1-GC#53 day1, which has no positive label), null shuffling validation labels
too 0.686 against 0.702 (p = 0.52). The null models are high (about 0.70) because the labels are
nearly constant within epochs, so the within-epoch shuffle keeps most of what separates epochs.

## rest-loss-accuracy-vs-null

> The resting-state decoder has lower validation loss and higher validation accuracy than the null
> models. (Expected: loss 0.54 against 0.68, accuracy 0.75 against 0.58; paired t-tests p < 0.05.)

Same models as rest-decoding-above-null (its seed, as this claim depends on it); accuracy at a
threshold of 0.5 at the epoch of lowest validation loss.

In [ ]:
for m, better in (("val_loss", "lower"), ("val_acc", "higher")):
    r = pw.paired(cv_main, m)
    ok = (r["model"] < r["null"]) if better == "lower" else (r["model"] > r["null"])
    print(f"{m}: model {r['model']:.3f}, null {r['null']:.3f}, t = {r['t']:.2f}, p = {r['p']:.3g}, "
          f"model {better} than null: {ok}")
print("Expected: loss 0.54 +/- 0.054 (model), 0.68 +/- 0.068 (null); accuracy 0.75 +/- 0.075, 0.58 +/- 0.058")
print("\nShare of negative labels in the validation folds (accuracy of always saying 'no dilation'):",
      np.round(1 - cv_main.pos_val, 3).tolist())
rows_s = []
for off, cv in cv_by_seed.items():
    a, b = pw.paired(cv, "val_loss"), pw.paired(cv, "val_acc")
    rows_s.append(dict(seed=f"claim+{off}", loss_model=a["model"], loss_null=a["null"], loss_p=a["p"],
                       acc_model=b["model"], acc_null=b["null"], acc_p=b["p"]))
print(pd.DataFrame(rows_s).round(4).to_string(index=False))
print("\nOther readings (claim's seed):")
print(vt[["reading", "val_loss_model", "val_loss_null", "val_loss_p", "val_acc_model", "val_acc_null", "val_acc_p"]].round(4).to_string(index=False))

**Result: not reproduced (resting on assumptions).** The validation loss is 0.622 against
0.623 for the null models (p = 0.963) and the accuracy 0.626 against 0.646 (p = 0.538); the paper's
loss 0.54 and accuracy 0.75 are not reached. No seed gives either test significant (loss p >= 0.16,
accuracy p >= 0.23). Among the other readings, only folds of recordings gives a significant loss
difference, in the wrong direction (model 0.662, null 0.620, p = 0.037). The every-frame reading
(R4 b, features notebook) is not significant either.

## rest-needs-0-1-hz

> Spontaneous pupil dilation could not be predicted above chance from cortical activity low-pass
> filtered at 2 Hz; a 0.1 Hz low-pass filter was required for significant decoding. The 0.1 Hz AUC
> did not differ significantly from that at any other cutoff tested (0.05, 0.2, 0.5, 1 and 2 Hz).
> (Expected AUC 0.86 +/- 0.086 at 0.1 Hz and 0.51 +/- 0.051 at 2 Hz.)

No reading combination reproduced rest-decoding-above-null above, so the criterion asks for the
cutoffs in all of them; they are run in the main combination and in R5 b (the only other one
that changes the samples cheaply), at this claim's seed. The 10 ICs are those selected at 0.1 Hz,
the paper's resting setting, for every cutoff [assumption: the paper does not say whether ICs were
reselected per cutoff].

In [ ]:
SEED3 = pw.seed_of("rest-needs-0-1-hz")
cut_rows, cut_cv = [], {}
for combo, kw in (("main", {}), ("R5 b", {"selection": "rule"})):
    for c in (0.05, 0.1, 0.2, 0.5, 1.0, 2.0):
        ds = pw.make_dataset(sessions, D["ics"], keep10, cutoff=c, **kw)
        cv = pw.run_cv(ds, SEED3)
        cut_cv[(combo, c)] = cv
        r = pw.paired(cv, "val_auc")
        cut_rows.append(dict(combo=combo, cutoff_hz=c, model_auc=r["model"], null_auc=r["null"], t=r["t"], p=r["p"],
                             above_null=bool(r["p"] < 0.05 and r["model"] > r["null"]),
                             median_auc=float(cv.model_val_auc.median())))
ct = pd.DataFrame(cut_rows)
print(ct.round(4).to_string(index=False))
for combo in ("main", "R5 b"):
    ref = cut_cv[(combo, 0.1)].model_val_auc
    others = [c for c in (0.05, 0.2, 0.5, 1.0, 2.0)]
    ps = [stats.ranksums(ref, cut_cv[(combo, c)].model_val_auc).pvalue for c in others]
    ph = multipletests(ps, method="holm")[1]
    print(f"{combo}: Wilcoxon rank-sum, 0.1 Hz against", dict(zip(others, np.round(ph, 4))), "(Holm-corrected)")

In [ ]:
rows_s = []
for off in (0, 1000, 2000, 3000, 4000):
    for c in (0.1, 2.0):
        cv = cut_cv[("main", c)] if off == 0 else pw.run_cv(pw.make_dataset(sessions, D["ics"], keep10, cutoff=c), SEED3 + off)
        r = pw.paired(cv, "val_auc")
        rows_s.append(dict(seed=f"claim+{off}", cutoff_hz=c, model_auc=r["model"], null_auc=r["null"], p=r["p"],
                           above_null=bool(r["p"] < 0.05 and r["model"] > r["null"])))
print(pd.DataFrame(rows_s).round(4).to_string(index=False))
print(f"\nelapsed {time.time() - started:.0f} s")

**Result: not reproduced (resting on assumptions).** At 0.1 Hz the models are not above their null
models (0.684 against 0.697, p = 0.63), under all five seeds (p = 0.20 to 0.98), so the claim that
0.1 Hz is needed for significant decoding is not supported in the main combination. The part of
absence holds there: at 2 Hz the models are not above null (0.677 against 0.702, p = 0.32; all five
seeds p >= 0.057), though the 2 Hz AUC is far from the paper's 0.51. No cutoff is above null in the
main combination (at 0.2 and 0.5 Hz the null is significantly higher, p = 0.037 and 0.047). With
the epoch selection rule (17 epochs) the models are above null at 0.5 Hz only (0.749 against
0.543, p = 0.039; 0.1 Hz 0.717 against 0.558, p = 0.089). As in the paper, the Holm-corrected
rank-sum tests of 0.1 Hz against every other cutoff are not significant (all p >= 0.24). The
cutoffs were not run in the every-frame reading (R4 b), the only one where the 0.1 Hz decoder is
above null.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| rest-decoding-above-null | partially reproduced (assumed; R4 b reading only, one seed) |
| rest-loss-accuracy-vs-null | not reproduced (assumed) |
| rest-needs-0-1-hz | not reproduced (assumed) |
| rest-ic-count-and-window | partially reproduced (assumed) |
| rest-vif-below-10 | not reproduced (assumed) |
| rest-shap-medial-anterior-first | not reproduced (assumed) |
| 18 claims on the paper's stimulation recordings | blocked: data missing |

**How the analysis differs from the paper's.** The paper does not say which three mice or which
resting sessions it used; the first three mice recorded, with their first three resting sessions,
are used. The ICs it kept "by visual inspection" are replaced by a repeatable rule (the most compact
maps). Frames are binned 2 x 2 (144 x 144) and the decomposition is made on a common atlas grid; the
SVD keeps 306 components, the rank of the corrected data, in place of 400. The GRU is written in
PyTorch with Keras' defaults, and SHAP is computed by integrated gradients from the mean input in
place of DeepExplainer. Failed pupil detections are replaced. Of the readings the claims file names,
the main combination and one change at a time were run, not every combination; the seed check was
run for the main reading of the decoding claims only. The JADE decomposition is not identical
across software environments (a run of the same code in another environment kept other ICs and gave
other AUCs, with the same outcomes except for the R4 b reading, which was not significant there);
the numbers here are those of this notebook's run.

**What was found about the dataset.** The files hold everything the resting analysis needs: frames
of both channels, pupil diameter and keypoints on the imaging clock, atlas ROIs and the atlas
transform. The excitation wavelengths of the two imaging planes are swapped in the metadata. The
deposited pupil diameter holds unflagged failed detections (values up to 2.9e21 pixels) in two of
the nine sessions read. Six of the 98 resting files hold no behaviour and so no pupil.